In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from utils.metric import metric
from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from lifelines import WeibullAFTFitter

np.random.seed(42)
pd.set_option("display.max_columns", 100)

In [2]:
HORIZONS = [12, 24, 48, 72]

In [3]:
train = pd.read_csv('../../data/train.csv')

In [4]:

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    eps = 1e-6

    dist = d["dist_min_ci_0_5h"].astype(float).clip(lower=1.0)  # meters
    dist_km = dist / 1000.0

    perims = d["num_perimeters_0_5h"].astype(float).clip(lower=0.0)
    dt_span = d["dt_first_last_0_5h"].astype(float).clip(lower=0.0)

    closing = d["closing_speed_m_per_h"].astype(float)
    closing_pos = closing.clip(lower=0.0)

    radial_rate = d["radial_growth_rate_m_per_h"].astype(float).clip(lower=0.0)
    along_track = d["along_track_speed"].astype(float) if "along_track_speed" in d.columns else 0.0

    align_abs = d["alignment_abs"].astype(float).clip(lower=0.0, upper=1.0)
    align_cos = d["alignment_cos"].astype(float) if "alignment_cos" in d.columns else 0.0

    area = d["area_first_ha"].astype(float).clip(lower=0.0)
    growth_rate_ha_h = d["area_growth_rate_ha_per_h"].astype(float).clip(lower=0.0)

    d["dist_km"] = dist_km
    d["log_dist"] = np.log1p(dist)
    d["sqrt_dist"] = np.sqrt(dist)

    d["inv_dist"] = 1.0 / (dist + 1.0)
    d["inv_dist_sq"] = d["inv_dist"] ** 2
    d["inv_dist_km"] = 1.0 / (dist_km + 0.05)

    d["zone_lt3km"] = (dist < 3000).astype(int)
    d["zone_3to5km"] = ((dist >= 3000) & (dist < 5000)).astype(int)
    d["zone_5to10km"] = ((dist >= 5000) & (dist < 10000)).astype(int)
    d["zone_ge10km"] = (dist >= 10000).astype(int)

    if "dist_fit_r2_0_5h" in d.columns:
        r2 = d["dist_fit_r2_0_5h"].astype(float).clip(lower=0.0, upper=1.0)
        d["dist_trend_reliable"] = (r2 > 0.6).astype(int)
        d["dist_r2"] = r2
    else:
        d["dist_trend_reliable"] = 0
        d["dist_r2"] = 0.0

    for col in ["dist_std_ci_0_5h", "dist_change_ci_0_5h", "dist_slope_ci_0_5h", "dist_accel_m_per_h2"]:
        if col in d.columns:
            d[col] = d[col].astype(float)

    if "dist_change_ci_0_5h" in d.columns:
        d["dist_change_km"] = d["dist_change_ci_0_5h"] / 1000.0
        d["dist_change_norm"] = d["dist_change_ci_0_5h"] / (dist + 1.0)

    if "dist_slope_ci_0_5h" in d.columns:
        d["dist_slope_norm"] = d["dist_slope_ci_0_5h"] / (dist + 1.0)

    if "dist_std_ci_0_5h" in d.columns:
        d["dist_std_norm"] = d["dist_std_ci_0_5h"] / (dist + 1.0)

    effective_closing = closing_pos + radial_rate
    d["effective_closing_speed"] = effective_closing

    d["eta_closing_h"] = np.where(closing_pos > 0.1, dist / (closing_pos + eps), 9999.0)
    d["eta_effective_h"] = np.where(effective_closing > 0.1, dist / (effective_closing + eps), 9999.0)

    d["log_eta_effective"] = np.log1p(np.clip(d["eta_effective_h"], 0, 9999))
    d["log_eta_closing"] = np.log1p(np.clip(d["eta_closing_h"], 0, 9999))

    for h in HORIZONS:
        d[f"eta_within_{h}h"] = (d["eta_effective_h"] <= float(h)).astype(int)
        d[f"eta_margin_{h}h"] = d["eta_effective_h"] - float(h)

    if isinstance(along_track, pd.Series):
        along_pos = along_track.astype(float).clip(lower=0.0)
        d["eta_alongtrack_h"] = np.where(along_pos > 0.1, dist / (along_pos + eps), 9999.0)
        d["log_eta_alongtrack"] = np.log1p(np.clip(d["eta_alongtrack_h"], 0, 9999))
    else:
        d["eta_alongtrack_h"] = 9999.0
        d["log_eta_alongtrack"] = np.log1p(9999.0)

    fire_radius_m = np.sqrt((area * 10000.0) / np.pi)
    d["fire_radius_m"] = fire_radius_m
    d["radius_to_dist"] = fire_radius_m / (dist + 1.0)

    d["dist_minus_radius"] = np.clip(dist - fire_radius_m, 1.0, None)
    d["log_dist_minus_radius"] = np.log1p(d["dist_minus_radius"])

    d["area_to_dist_km"] = area / (dist_km + 0.1)
    d["growth_to_dist_km"] = growth_rate_ha_h / (dist_km + 0.1)

    if "radial_growth_m" in d.columns:
        d["radial_growth_m"] = d["radial_growth_m"].astype(float).clip(lower=0.0)
        d["radial_to_dist"] = d["radial_growth_m"] / (dist + 1.0)

    d["threat_pressure"] = align_abs * effective_closing / (np.log1p(dist) + eps)
    d["alignment_x_closing"] = align_abs * closing_pos
    d["alignment_x_effective"] = align_abs * effective_closing

    if "cross_track_component" in d.columns:
        ctc = d["cross_track_component"].astype(float)
        d["cross_track_abs"] = np.abs(ctc)
        d["cross_track_norm"] = np.abs(ctc) / (dist + 1.0)

    if "spread_bearing_sin" in d.columns and "spread_bearing_cos" in d.columns:
        sb_sin = d["spread_bearing_sin"].astype(float)
        sb_cos = d["spread_bearing_cos"].astype(float)
        d["bearing_strength"] = np.sqrt(sb_sin**2 + sb_cos**2) 

    d["has_movement"] = (perims > 1).astype(int)
    d["perim_density"] = perims / (dt_span + 0.25) 
    d["short_window"] = (dt_span < 0.5).astype(int)

    if "low_temporal_resolution_0_5h" in d.columns:
        d["low_temporal_resolution_0_5h"] = d["low_temporal_resolution_0_5h"].astype(int)

    hour = d["event_start_hour"].astype(float) if "event_start_hour" in d.columns else 0.0
    d["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    d["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)

    dow = d["event_start_dayofweek"].astype(float) if "event_start_dayofweek" in d.columns else 0.0
    d["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
    d["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)

    month = d["event_start_month"].astype(float) if "event_start_month" in d.columns else 1.0
    d["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    d["month_cos"] = np.cos(2 * np.pi * month / 12.0)

    d = d.replace([np.inf, -np.inf], np.nan).fillna(0)

    return d



In [5]:
def make_strata(t, e):
    strata = np.zeros(len(t))
    strata[e == 1] = 1
    return strata

In [6]:
train_fe = build_features(train)

In [7]:
def enforce_monotone(P):
    P = np.clip(P, 0.0, 1.0)
    P[:, 1] = np.maximum(P[:, 1], P[:, 0])
    P[:, 2] = np.maximum(P[:, 2], P[:, 1])
    P[:, 3] = np.maximum(P[:, 3], P[:, 2])
    return np.clip(P, 0.0, 1.0)

def masked_brier_for_horizon(p, t, e, H):
    mask = []
    y = []
    for i in range(len(t)):
        if e[i] == 1 and t[i] <= H:
            mask.append(True); y.append(1)
        elif t[i] > H:
            mask.append(True); y.append(0)
        else:
            mask.append(False); y.append(0)
    mask = np.array(mask, dtype=bool)
    y = np.array(y, dtype=int)
    if mask.sum() == 0:
        return 0.0
    return np.mean((p[mask] - y[mask])**2)

In [8]:
drop_cols = ["event_id", "event", "time_to_hit_hours"]

X = train_fe.drop(columns=drop_cols)
t = train_fe["time_to_hit_hours"].values.astype(float)
e = train_fe["event"].values.astype(int)

strata = make_strata(t, e)

In [9]:
def km_censor_survival(times, censor_event):
    order = np.argsort(times)
    times = times[order]
    censor_event = censor_event[order]

    uniq = np.unique(times)
    n = len(times)

    at_risk = n
    G = 1.0
    G_map = {}

    for tt in uniq:
        m = np.sum(times == tt)
        d = np.sum((times == tt) & (censor_event == 1))

        if at_risk > 0:
            G *= (1.0 - d / at_risk)

        G_map[tt] = max(G, 1e-6)
        at_risk -= m

    uniq_times = np.array(sorted(G_map.keys()))
    G_vals = np.array([G_map[u] for u in uniq_times])
    return uniq_times, G_vals


def G_of_t(t_query, uniq_times, G_vals):
    t_query = np.asarray(t_query)
    idx = np.searchsorted(uniq_times, t_query, side="right") - 1
    out = np.ones_like(t_query, dtype=float)
    ok = idx >= 0
    out[ok] = G_vals[idx[ok]]
    return np.clip(out, 1e-6, 1.0)

In [10]:
def make_ipcw_data(X, t, e, H, uniq_times, G_vals, weight_cap=30.0):

    y = ((e == 1) & (t <= H)).astype(int)

    mask = ((e == 1) & (t <= H)) | (t > H)

    t_clip = np.minimum(t, H)

    G_t = G_of_t(t_clip, uniq_times, G_vals)

    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask

In [11]:
def bagged_lgb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):

    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = lgb.LGBMClassifier(
            n_estimators=2000,
            learning_rate=0.02,
            num_leaves=15,
            max_depth=3,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=8.0,
            objective="binary",
            random_state=seed,
            verbose=-1
        )

        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds

In [12]:
def make_tail_data(X, t, e, H0, H1, uniq_times, G_vals, weight_cap=50.0):
    y = ((e == 1) & (t > H0) & (t <= H1)).astype(int)

    mask = ((e == 1) & (t > H0) & (t <= H1)) | (t > H1)

    t_clip = np.minimum(t, H1)
    G_t = G_of_t(t_clip, uniq_times, G_vals)
    w = 1.0 / G_t
    w = np.clip(w, 1.0, weight_cap)

    return X[mask], y[mask], w[mask], mask

In [13]:
def run_pipeline(X_tr, t_tr, e_tr, X_va, t_va, e_va):

    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    H = 12
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    p12_full = np.zeros(len(X_va))
    p12_full[mask_va] = bagged_lgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=10)
    H = 24
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
        X_tr, t_tr, e_tr, H, uniq_times, G_vals
    )
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(
        X_va, t_va, e_va, H, uniq_times, G_vals
    )

    p24_full = np.zeros(len(X_va))
    p24_full[mask_va] = bagged_lgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=10)

    X48_tr, y48_tr, w48_tr, m48_tr = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    tail_feats = [
        "dist_km",
        "eta_effective_h",
        "effective_closing_speed",
        "alignment_abs",
        "has_movement"
    ]

    lr_48 = LogisticRegression(
        C=0.2,
        solver="lbfgs",
        max_iter=2000
    )

    lr_48.fit(
        X48_tr[tail_feats],
        y48_tr,
        sample_weight=w48_tr
    )

    p48_tail_full = np.zeros(len(X_va))
    p48_tail_full[m48_va] = lr_48.predict_proba(
        X48_va[tail_feats]
    )[:, 1]

    p48_full = p24_full + (1 - p24_full) * p48_tail_full

    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48_full + (1 - p48_full) * alpha

    P = np.zeros((len(X_va), 4))
    P[:, 0] = p12_full
    P[:, 1] = p24_full
    P[:, 2] = p48_full
    P[:, 3] = p72

    P = enforce_monotone(P)

    b12 = masked_brier_for_horizon(P[:,0], t_va, e_va, 12)
    b24 = masked_brier_for_horizon(P[:,1], t_va, e_va, 24)
    b48 = masked_brier_for_horizon(P[:,2], t_va, e_va, 48)
    b72 = masked_brier_for_horizon(P[:,3], t_va, e_va, 72)

    print("Brier 12:", b12)
    print("Brier 24:", b24)
    print("Brier 48:", b48)
    print("Brier 72:", b72)

    return metric(P, t_va, e_va), P

In [14]:
def bagged_xgb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):
    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = xgb.XGBClassifier(
                n_estimators=2500,
                learning_rate=0.03,
                max_depth=4,
                min_child_weight=1,
                subsample=0.9,
                colsample_bytree=0.6, 
                reg_alpha=0.5,
                reg_lambda=1.0,
                gamma=2.0,  
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=seed,
        )
        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds


def run_pipeline_xgb(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    H = 12
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

    p12_full = np.zeros(len(X_va))
    p12_full[mask_va] = bagged_xgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=8)

    H = 24
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

    p24_full = np.zeros(len(X_va))
    p24_full[mask_va] = bagged_xgb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=8)

    X48_tr, y48_tr, w48_tr, _ = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    tail_feats = [
        "dist_km",
        "eta_effective_h",
        "effective_closing_speed",
        "alignment_abs",
        "has_movement"
    ]

    p48_tail_full = np.zeros(len(X_va))
    p48_tail_full[m48_va] = bagged_xgb_predict(
        X48_tr[tail_feats], y48_tr, w48_tr,
        X48_va[tail_feats],
        n_seeds=10
    )

    p48_full = p24_full + (1 - p24_full) * p48_tail_full

    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48_full + (1 - p48_full) * alpha
    
    P = np.zeros((len(X_va), 4))
    P[:, 0] = p12_full
    P[:, 1] = p24_full
    P[:, 2] = p48_full
    P[:, 3] = p72

    P = enforce_monotone(P)

    return metric(P, t_va, e_va), P

In [15]:
def bagged_cb_predict(X_tr, y_tr, w_tr, X_va, n_seeds=5):
    preds = np.zeros(len(X_va))

    for seed in range(42, 42 + n_seeds):
        model = CatBoostClassifier(
            iterations=4000,
            learning_rate=0.03,
            depth=4,
            l2_leaf_reg=8.0,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=seed,
            verbose=False
        )
        model.fit(X_tr, y_tr, sample_weight=w_tr)
        preds += model.predict_proba(X_va)[:, 1]

    return preds / n_seeds

In [16]:
def run_pipeline_cat(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    H = 12
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)
    p12 = np.zeros(len(X_va)); p12[mask_va] = bagged_cb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=5)

    H = 24
    Xh_tr, yh_tr, wh_tr, _ = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)
    Xh_va, yh_va, wh_va, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)
    p24 = np.zeros(len(X_va)); p24[mask_va] = bagged_cb_predict(Xh_tr, yh_tr, wh_tr, Xh_va, n_seeds=5)

    X48_tr, y48_tr, w48_tr, _ = make_tail_data(X_tr, t_tr, e_tr, 24, 48, uniq_times, G_vals)
    X48_va, y48_va, w48_va, m48_va = make_tail_data(X_va, t_va, e_va, 24, 48, uniq_times, G_vals)

    tail_feats = ["dist_km","eta_effective_h","effective_closing_speed","alignment_abs","has_movement"]
    p48_tail = np.zeros(len(X_va))
    p48_tail[m48_va] = bagged_cb_predict(
        X48_tr[tail_feats], y48_tr, w48_tr,
        X48_va[tail_feats],
        n_seeds=5
    )
    p48 = p24 + (1 - p24) * p48_tail

    late_events = np.sum((e_tr == 1) & (t_tr > 48))
    survive_48 = np.sum(t_tr > 48)
    alpha = (late_events + 1) / (survive_48 + 2)

    p72 = p48 + (1 - p48) * alpha

    P = np.vstack([p12, p24, p48, p72]).T
    P = enforce_monotone(P)
    return metric(P, t_va, e_va), P

In [17]:
def fit_xgb_cox(X_tr, t_tr, e_tr, X_va, t_va, e_va, seed=42):
    params = {
        "objective": "survival:cox",
        "eta": 0.02,
        "max_depth": 4,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "min_child_weight": 2,
        "lambda": 1.0,
        "alpha": 0.0,
        "seed": seed,
        "tree_method": "hist",
        "eval_metric": "cox-nloglik",
    }
    y_tr = t_tr.astype(float).copy()
    y_tr[e_tr == 0] *= -1.0

    y_va = t_va.astype(float).copy()
    y_va[e_va == 0] *= -1.0

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_va, label=y_va)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=5000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=200,
        verbose_eval=False
    )

    return model

def predict_cox_risk(model, X):
    d = xgb.DMatrix(X)
    risk = model.predict(d)  
    return risk

def run_pipeline_cox_risk(X_tr, t_tr, e_tr, X_va, t_va, e_va):

    model = fit_xgb_cox(
        X_tr, t_tr, e_tr,
        X_va, t_va, e_va,
        seed=42
    )

    risk_va = predict_cox_risk(model, X_va)
    return risk_va


In [18]:
def iso_map_risk_to_prob(risk_tr, X_tr, t_tr, e_tr, H, uniq_times, G_vals):
    Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(X_tr, t_tr, e_tr, H, uniq_times, G_vals)

    r_tr = risk_tr[mask_tr]
    y_tr = yh_tr
    w_tr = wh_tr

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(r_tr, y_tr, sample_weight=w_tr)
    return iso, mask_tr

def cox_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    risk_tr = predict_cox_risk(model, X_tr)
    risk_va = predict_cox_risk(model, X_va)

    P = np.zeros((len(X_va), 4))

    for j, H in enumerate([12,24,48,72]):
        iso, mask_tr = iso_map_risk_to_prob(risk_tr, X_tr, t_tr, e_tr, H, uniq_times, G_vals)

        _, _, _, mask_va = make_ipcw_data(X_va, t_va, e_va, H, uniq_times, G_vals)

        p = np.zeros(len(X_va))
        p[mask_va] = iso.predict(risk_va[mask_va])
        P[:, j] = p

    return enforce_monotone(P)

In [19]:
def run_pipeline_cox(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    model = fit_xgb_cox(
        X_tr, t_tr, e_tr,
        X_va, t_va, e_va,
        seed=42
    )
    P = cox_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va)
    return metric(P, t_va, e_va), P

In [20]:
def fit_xgb_aft(X_tr, t_tr, e_tr, X_va, t_va, e_va, seed=42):
    lower_tr = np.log1p(t_tr.astype(float))
    upper_tr = np.log1p(t_tr.astype(float))
    upper_tr[e_tr == 0] = np.inf

    dtrain = xgb.DMatrix(X_tr)
    dtrain.set_float_info("label_lower_bound", lower_tr)
    dtrain.set_float_info("label_upper_bound", upper_tr)

    lower_va = np.log1p(t_va.astype(float))
    upper_va = np.log1p(t_va.astype(float))
    upper_va[e_va == 0] = np.inf

    dvalid = xgb.DMatrix(X_va)
    dvalid.set_float_info("label_lower_bound", lower_va)
    dvalid.set_float_info("label_upper_bound", upper_va)

    params = {
        "objective": "survival:aft",
        "aft_loss_distribution": "extreme",
        "aft_loss_distribution_scale": 1.5,
        "eta": 0.02,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.85,
        "colsample_bytree": 0.8,
        "lambda": 3.0,
        "alpha": 0.5,
        "tree_method": "hist",
        "seed": seed,
    }

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=5000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=200,
        verbose_eval=False
    )
    return model

def predict_aft_risk(model, X):
    d = xgb.DMatrix(X)
    pred_time = model.predict(d) 
    risk = -pred_time
    return risk

def run_pipeline_aft_risk(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    model = fit_xgb_aft(X_tr, t_tr, e_tr, X_va, t_va, e_va, seed=42)
    return predict_aft_risk(model, X_va)

In [21]:
def aft_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va):
    c_tr = (e_tr == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t_tr, c_tr)

    risk_tr = predict_aft_risk(model, X_tr)
    risk_va = predict_aft_risk(model, X_va)

    P = np.zeros((len(X_va), 4))

    for j, H in enumerate([12,24,48,72]):

        Xh_tr, yh_tr, wh_tr, mask_tr = make_ipcw_data(
            X_tr, t_tr, e_tr, H, uniq_times, G_vals
        )

        r_tr = risk_tr[mask_tr]
        y_tr = yh_tr
        w_tr = wh_tr

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(r_tr, y_tr, sample_weight=w_tr)

        _, _, _, mask_va = make_ipcw_data(
            X_va, t_va, e_va, H, uniq_times, G_vals
        )

        p = np.zeros(len(X_va))
        p[mask_va] = iso.predict(risk_va[mask_va])
        P[:, j] = p

    return enforce_monotone(P)


def run_pipeline_aft(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    model = fit_xgb_aft(
        X_tr, t_tr, e_tr,
        X_va, t_va, e_va,
        seed=42
    )
    P = aft_probs_from_risk(model, X_tr, t_tr, e_tr, X_va, t_va, e_va)
    return metric(P, t_va, e_va), P

In [22]:
def run_pipeline_lgb_wrapper(X_tr, t_tr, e_tr, X_va, t_va, e_va):
    (c, b, s), P = run_pipeline(X_tr, t_tr, e_tr, X_va, t_va, e_va)
    return (c, b, s), P


def collect_oof_preds(X, t, e, strata, skf):
    oof_lgb = np.zeros((len(X), 4))
    oof_xgb = np.zeros((len(X), 4))
    oof_cat = np.zeros((len(X), 4))
    oof_cox = np.zeros((len(X), 4))
    oof_aft = np.zeros((len(X), 4))

    scores = {"lgb":[], "xgb":[], "cat":[], "cox":[], "aft":[]}

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, strata)):
        print("Fold:", fold)

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        t_tr, t_va = t[tr_idx], t[va_idx]
        e_tr, e_va = e[tr_idx], e[va_idx]

        (c_l,b_l,s_l), P_lgb = run_pipeline_lgb_wrapper(X_tr, t_tr, e_tr, X_va, t_va, e_va)
        (c_x,b_x,s_x), P_xgb = run_pipeline_xgb(X_tr, t_tr, e_tr, X_va, t_va, e_va)
        (c_c,b_c,s_c), P_cat = run_pipeline_cat(X_tr, t_tr, e_tr, X_va, t_va, e_va)
        (c_k,b_k,s_k), P_cox = run_pipeline_cox(X_tr, t_tr, e_tr, X_va, t_va, e_va)
        (c_a,b_a,s_a), P_aft = run_pipeline_aft(X_tr, t_tr, e_tr, X_va, t_va, e_va)

        oof_lgb[va_idx] = P_lgb
        oof_xgb[va_idx] = P_xgb
        oof_cat[va_idx] = P_cat
        oof_cox[va_idx] = P_cox
        oof_aft[va_idx] = P_aft

        scores["lgb"].append(s_l)
        scores["xgb"].append(s_x)
        scores["cat"].append(s_c)
        scores["cox"].append(s_k)
        scores["aft"].append(s_a)

        print("fold scores:", s_l, s_x, s_c, s_k, s_a)

    for k,v in scores.items():
        print(k, "CV:", np.mean(v), np.std(v))

    return oof_lgb, oof_xgb, oof_cat, oof_cox, oof_aft

In [23]:
def collect_oof_risks(X, t, e, strata, skf):
    oof_cox_risk = np.zeros(len(X))
    oof_aft_risk = np.zeros(len(X))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, strata)):
        print("Risk Fold:", fold)

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        t_tr, t_va = t[tr_idx], t[va_idx]
        e_tr, e_va = e[tr_idx], e[va_idx]

        risk_cox = run_pipeline_cox_risk(
                        X_tr, t_tr, e_tr,
                        X_va, t_va, e_va
                    )
        risk_aft = run_pipeline_aft_risk(X_tr, t_tr, e_tr,
                        X_va, t_va, e_va)

        oof_cox_risk[va_idx] = risk_cox
        oof_aft_risk[va_idx] = risk_aft

    return oof_cox_risk, oof_aft_risk

In [24]:
def blend_per_horizon(models, W):
    """
    W shape: (4, n_models)
    """
    n = models[0].shape[0]
    P = np.zeros((n, 4), dtype=float)
    for j in range(4):
        for m_idx, Pi in enumerate(models):
            P[:, j] += W[j, m_idx] * Pi[:, j]
    return enforce_monotone(P)

In [25]:
def optimize_risk_blend(risk1, risk2, t, e):
    best = (-1e9, None)

    for w in np.linspace(0, 1, 41):
        blended = w * risk1 + (1 - w) * risk2
        P_fake = np.zeros((len(t), 4))
        P_fake[:, 0] = blended
        P_fake[:, 1] = blended
        P_fake[:, 2] = blended
        P_fake[:, 3] = blended

        c, b, s = metric(P_fake, t, e)

        if c > best[0]:
            best = (c, w)

    print("Best risk blend weight:", best[1], "C-index:", best[0])
    return best[1]

In [26]:
def risk_to_prob_matrix(risk, X, t, e):
    c_tr = (e == 0).astype(int)
    uniq_times, G_vals = km_censor_survival(t, c_tr)

    P = np.zeros((len(t), 4))

    for j, H in enumerate([12,24,48,72]):
        Xh, yh, wh, mask = make_ipcw_data(X, t, e, H, uniq_times, G_vals)

        r_tr = risk[mask]
        y_tr = yh
        w_tr = wh

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(r_tr, y_tr, sample_weight=w_tr)

        p = np.zeros(len(t))
        p[mask] = iso.predict(risk[mask])
        P[:, j] = p

    return enforce_monotone(P)

In [27]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb, oof_xgb, oof_cat, oof_cox, oof_aft = collect_oof_preds(X, t, e, strata, skf)
oof_cox_risk, oof_aft_risk = collect_oof_risks(X, t, e, strata, skf)

w_risk = optimize_risk_blend(oof_cox_risk, oof_aft_risk, t, e)
blended_risk = w_risk * oof_cox_risk + (1 - w_risk) * oof_aft_risk

P_risk = risk_to_prob_matrix(blended_risk, X, t, e)
print("Risk-based OOF:", metric(P_risk, t, e))

print("Cox risk C-index:", metric(np.vstack([oof_cox_risk]*4).T, t, e)[0])
print("AFT risk C-index:", metric(np.vstack([oof_aft_risk]*4).T, t, e)[0])
print("Blended risk C-index:", metric(np.vstack([blended_risk]*4).T, t, e)[0])

models = [oof_lgb, oof_xgb, oof_cat, P_risk]
names  = ["lgb", "xgb", "cat", "risk"]

HORIZONS = [12, 24, 48, 72]

Fold: 0
Brier 12: 0.07749164638325122
Brier 24: 0.03472230395063873
Brier 48: 0.013113552943398443
Brier 72: 0.020731835050885824
fold scores: 0.964423095725788 0.9693792719186842 0.9755793930445901 0.9663295703449256 0.9710396405785302
Fold: 1
Brier 12: 0.05554650901651496
Brier 24: 0.05058803043659643
Brier 48: 0.025629527419886944
Brier 72: 0.01959765230269615
fold scores: 0.9583996995771014 0.9557058408422359 0.9703879085952682 0.9546916782461015 0.96712374322758
Fold: 2
Brier 12: 0.041967553247845235
Brier 24: 0.04298036368800924
Brier 48: 0.03391956454852179
Brier 72: 0.009029453885514535
fold scores: 0.9661726390069236 0.9658109615826067 0.9529709046867896 0.9544506517690875 0.970315429386384
Fold: 3
Brier 12: 0.06645646962248723
Brier 24: 0.02643550623153769
Brier 48: 0.038005585604326216
Brier 72: 0.02056739611172452
fold scores: 0.9553498955042208 0.9612511973729996 0.969883255782404 0.956226362625139 0.9622001687692427
Fold: 4
Brier 12: 0.0601638663862233
Brier 24: 0.0078459

In [28]:
def coordinate_descent_per_horizon(models, t, e, grid_step=0.1, n_passes=3):

    n_models = len(models)
    grid = np.arange(0, 1 + 1e-9, grid_step)

    W = np.ones((4, n_models), dtype=float) / n_models

    best_global = metric(blend_per_horizon(models, W), t, e)[2]
    print("init score:", best_global)
    print("init W:\n", W)

    for it in range(n_passes):
        print("\n=== pass", it, "===")

        for j in range(4):

            best_j = (-1e9, None)

            for w0 in grid:
                for w1 in grid:
                    for w2 in grid:

                        w3 = 1 - (w0 + w1 + w2)

                        if w3 < -1e-9:
                            continue
                        if w3 < 0:
                            w3 = 0.0

                        w = np.array([w0, w1, w2, w3], dtype=float)

                        if abs(w.sum() - 1.0) > 1e-6:
                            continue

                        W_trial = W.copy()
                        W_trial[j] = w

                        P_trial = blend_per_horizon(models, W_trial)
                        c, b, s = metric(P_trial, t, e)

                        if s > best_j[0]:
                            best_j = (s, w.copy())

            W[j] = best_j[1]

            print(
                f"H{HORIZONS[j]} best w:",
                dict(zip(names, W[j])),
                "score:", best_j[0]
            )

        s_now = metric(blend_per_horizon(models, W), t, e)[2]
        print("score after pass:", s_now)

        if s_now <= best_global + 1e-7:
            print("no improvement, stopping early")
            break

        best_global = s_now

    return W

In [29]:
def sharpen_p12(P, power):
    Q = P.copy()
    Q[:, 0] = np.clip(Q[:, 0] ** power, 0, 1)
    return enforce_monotone(Q)

def tune_p12_power(P, t, e, powers=None):
    if powers is None:
        powers = [0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95,1.00]

    best = (-1e9, None, None)
    for p in powers:
        Q = sharpen_p12(P, p)
        c, b, s = metric(Q, t, e)
        if s > best[0]:
            best = (s, p, (c, b))
    print("best p12 power:", best[1], "score:", best[0], "c,b:", best[2])
    return best[1]

def post_blend_isotonic(P, t, e):
    """
    Isotonic per horizon for 24/48/72 (leave 12 alone)
    using censor-aware mask:
      keep if (event and t<=H) OR (t>H)
    """
    Q = P.copy()
    for j, H in zip([1, 2, 3], [24, 48, 72]):
        mask = ((e == 1) & (t <= H)) | (t > H)
        y = ((e == 1) & (t <= H)).astype(int)

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(P[mask, j], y[mask])
        Q[:, j] = iso.predict(P[:, j])

    return enforce_monotone(Q)

In [30]:
W = coordinate_descent_per_horizon(models, t, e, grid_step=0.1, n_passes=3)

print("\nFinal per-horizon weights:")
for j in range(4):
    print(f"H{HORIZONS[j]}:", dict(zip(names, W[j])))

P = blend_per_horizon(models, W)

print("\nOOF after per-horizon blend:", metric(P, t, e))

best_p = tune_p12_power(P, t, e)
P = sharpen_p12(P, best_p)

post_isos = {}

P_cal = P.copy()

for j, H in zip([1, 2, 3], [24, 48, 72]):
    mask = ((e == 1) & (t <= H)) | (t > H)
    y = ((e == 1) & (t <= H)).astype(int)

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(P[mask, j], y[mask])

    P_cal[:, j] = iso.predict(P[:, j])
    post_isos[j] = iso

P = enforce_monotone(P_cal)

print("\nFINAL blended OOF:")
c_index, brier, score = metric(P, t, e)
print("C-index:", c_index)
print("Brier:", brier)
print("Final score:", score)

init score: 0.9700061689377875
init W:
 [[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]

=== pass 0 ===
H12 best w: {'lgb': np.float64(0.0), 'xgb': np.float64(0.30000000000000004), 'cat': np.float64(0.0), 'risk': np.float64(0.7)} score: 0.9701820891825153
H24 best w: {'lgb': np.float64(0.0), 'xgb': np.float64(0.0), 'cat': np.float64(0.1), 'risk': np.float64(0.9)} score: 0.9713901645148271
H48 best w: {'lgb': np.float64(0.0), 'xgb': np.float64(0.0), 'cat': np.float64(0.0), 'risk': np.float64(1.0)} score: 0.9731760098662728
H72 best w: {'lgb': np.float64(0.0), 'xgb': np.float64(0.0), 'cat': np.float64(0.0), 'risk': np.float64(1.0)} score: 0.9737422938968588
score after pass: 0.9737422938968588

=== pass 1 ===
H12 best w: {'lgb': np.float64(0.1), 'xgb': np.float64(0.6000000000000001), 'cat': np.float64(0.0), 'risk': np.float64(0.29999999999999993)} score: 0.9739855288991495
H24 best w: {'lgb': np.float64(0.0), 'xgb': np.float64(0.0), 'cat': np.

In [31]:
from scipy.stats import spearmanr

spearmanr(oof_cox_risk, oof_aft_risk)

SignificanceResult(statistic=np.float64(0.9229686458531863), pvalue=np.float64(3.2537534211340612e-83))

In [32]:
from scipy.stats import spearmanr

print("cox vs lgb:", spearmanr(oof_cox_risk, oof_lgb[:,0]))
print("cox vs xgb:", spearmanr(oof_cox_risk, oof_xgb[:,0]))
print("cox vs cat:", spearmanr(oof_cox_risk, oof_cat[:,0]))

cox vs lgb: SignificanceResult(statistic=np.float64(0.8006432820101032), pvalue=np.float64(1.7633483125023614e-45))
cox vs xgb: SignificanceResult(statistic=np.float64(0.8665798061100815), pvalue=np.float64(4.477152569148907e-61))
cox vs cat: SignificanceResult(statistic=np.float64(0.7950340134860624), pvalue=np.float64(1.9840095694034233e-44))


In [33]:
print("LGB:", metric(oof_lgb, t, e))
print("XGB:", metric(oof_xgb, t, e))
print("CAT:", metric(oof_cat, t, e))
print("COX:", metric(oof_cox, t, e))
print("AFT:", metric(oof_aft, t, e))

LGB: (0.9343549614172373, np.float64(0.024923625478509882), np.float64(0.9628599505902142))
XGB: (0.9327247038365395, np.float64(0.02441968364200612), np.float64(0.9627236326015575))
CAT: (0.9290294533202913, np.float64(0.016252796369526912), np.float64(0.9673318785374185))
COX: (0.9187588305618954, np.float64(0.023395645289813735), np.float64(0.959250697465699))
AFT: (0.9365829801108575, np.float64(0.015789106733906856), np.float64(0.9699225193195224))


## Important metrics

In [34]:
w_risk

np.float64(0.0)

In [35]:
W

array([[0.1, 0.6, 0. , 0.3],
       [0. , 0. , 0.1, 0.9],
       [0. , 0. , 0. , 1. ],
       [0. , 0. , 0. , 1. ]])

In [36]:
best_p

1.0

In [37]:
import joblib

bundle = {
    "W": W,
    "w_risk": w_risk,
    "best_p": best_p,
    "post_isos": post_isos
}

joblib.dump(bundle, "stack_bundle.joblib")
print("Saved stack_bundle.joblib")

Saved stack_bundle.joblib
